# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available Record Sets
record_sets = list(dataset.record_sets.keys())
print("Available Record Sets (@id):")
for rset in record_sets:
    print(f"  - {rset}")
    recset_meta = dataset.record_sets[rset]
    print(f"    Name: {getattr(recset_meta, 'name', 'N/A')}")
    print(f"    Description: {getattr(recset_meta, 'description', 'N/A')}")
    # List field @ids in the record set
    field_ids = [f['@id'] for f in getattr(recset_meta, 'fields', [])]
    print("    Fields (@id):")
    for fid in field_ids:
        print(f"      - {fid}")

# For large datasets, let's select the main record set for further exploration. We'll use the first one found.
main_record_set = record_sets[0] if record_sets else None
print(f"\nMain record set selected for further analysis: {main_record_set}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set in record_sets:
    print(f"Loading records for record set: {record_set}")
    records_iter = dataset.records(record_set=record_set)
    records = list(records_iter)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"  Columns for {record_set}: {df.columns.tolist()}")
        display(df.head(3))
    else:
        print(f"  No records found for {record_set}.")

# Proceed with main_record_set for further steps


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Pick a numeric field from the main record set for demonstration.
numeric_candidates = []
if main_record_set and main_record_set in dataframes:
    df = dataframes[main_record_set]
    # Try common numeric column names if possible
    preferred_numeric_fields = ['cr:field.Molecular_Weight', 'cr:field.Abundance', 'cr:field.Coverage_Percent', 'cr:field.Unique_Peptide_Count']
    for col in df.columns:
        if col in preferred_numeric_fields or pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidates.append(col)
    print(f"Numeric field candidates: {numeric_candidates}")
    # Select one
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        # Remove rows with missing or non-numeric
        df = df.copy()
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean() if df[numeric_field].mean() is not None else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field
        group_candidates = [c for c in df.columns if c != numeric_field and df[c].nunique() < df.shape[0]//2]
        print(f"Potential group-by fields: {group_candidates}")
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields found in the main record set.")
else:
    print("No records or main record set available for EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set and main_record_set in dataframes and numeric_candidates:
    df = dataframes[main_record_set]
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='teal')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field was found earlier, plot mean numeric_field per group
    if 'group_field' in locals():
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field, palette='viridis')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: numeric field or records unavailable.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the dataset metadata and discovered available record sets and fields using their `@id`s.
- We extracted tabular data, identified numeric fields such as protein abundance or molecular weight, and performed basic filtering and normalization.
- Data was grouped and visualized, highlighting key distributions and categorical breakdowns.
- The `mlcroissant` library enabled transparent, schema-driven data extraction guided by Croissant `@id` referencing.